# AeroNetra — Multi-Model Inference Comparison

**Purpose:** Run all trained models on test images side by side. Compare detection quality, speed, and counting accuracy visually.

**Kaggle Setup:**
1. Add datasets:
   - `aeronetra-visdrone-yolo` (output from notebook 01)
   - `aeronetra-trained-weights` (output from notebook 02)
2. Accelerator: **GPU T4 x2** (inference is faster on GPU)
3. Internet: ON (if ultralytics needs install)

**Outputs:** Annotated images, detection JSON/CSV, speed benchmarks.

In [ ]:
# ============================================================
# Cell 1: Install & setup
# ============================================================
!pip install -q ultralytics

import torch
import cv2
import numpy as np
import time
import json
import csv
from pathlib import Path
from dataclasses import dataclass, field

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

In [ ]:
# ============================================================
# Cell 2: Data structures (inlined from aeronetra.detection.types)
# ============================================================

@dataclass
class BoundingBox:
    xmin: float
    ymin: float
    xmax: float
    ymax: float

    def __post_init__(self):
        if self.xmin > self.xmax:
            raise ValueError(f"xmin ({self.xmin}) > xmax ({self.xmax})")
        if self.ymin > self.ymax:
            raise ValueError(f"ymin ({self.ymin}) > ymax ({self.ymax})")

    @property
    def xyxy(self): return (self.xmin, self.ymin, self.xmax, self.ymax)
    @property
    def width(self): return self.xmax - self.xmin
    @property
    def height(self): return self.ymax - self.ymin
    @property
    def area(self): return max(0.0, self.width) * max(0.0, self.height)
    @property
    def center(self): return (self.xmin + self.width / 2, self.ymin + self.height / 2)


@dataclass
class Detection:
    box: BoundingBox
    class_id: int
    class_name: str
    confidence: float
    source_model: str = "unknown"
    image_id: str = "unknown"


@dataclass
class ModelPrediction:
    detections: list[Detection] = field(default_factory=list)
    image_width: int = 0
    image_height: int = 0
    inference_time_ms: float = 0.0

    def filter_by_confidence(self, threshold: float) -> "ModelPrediction":
        filtered = [d for d in self.detections if d.confidence >= threshold]
        return ModelPrediction(filtered, self.image_width, self.image_height, self.inference_time_ms)

    def filter_by_class(self, allowed_classes: list[int]) -> "ModelPrediction":
        filtered = [d for d in self.detections if d.class_id in allowed_classes]
        return ModelPrediction(filtered, self.image_width, self.image_height, self.inference_time_ms)


@dataclass
class CountSummary:
    image_id: str
    total_vehicles: int
    class_counts: dict[str, int]
    model_name: str


print("Data structures defined.")

In [ ]:
# ============================================================
# Cell 3: Drawing & counting helpers (inlined from aeronetra.counting)
# ============================================================

def count_vehicles(detections: list[Detection]) -> tuple[int, dict]:
    total = len(detections)
    class_counts = {}
    for d in detections:
        class_counts[d.class_name] = class_counts.get(d.class_name, 0) + 1
    return total, class_counts


def draw_detections(image: np.ndarray, detections: list[Detection],
                    color=(0, 255, 0), thickness=2) -> np.ndarray:
    out = image.copy()
    for d in detections:
        x1, y1 = int(d.box.xmin), int(d.box.ymin)
        x2, y2 = int(d.box.xmax), int(d.box.ymax)
        cv2.rectangle(out, (x1, y1), (x2, y2), color, thickness)
        label = f"{d.class_name} {d.confidence:.2f}"
        (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        cv2.rectangle(out, (x1, y1 - h - 4), (x1 + w, y1), color, -1)
        cv2.putText(out, label, (x1, y1 - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
    return out


def draw_count_summary(image: np.ndarray, summary: CountSummary) -> np.ndarray:
    out = image.copy()
    lines = [f"Total: {summary.total_vehicles}"]
    for cls, cnt in summary.class_counts.items():
        lines.append(f"{cls}: {cnt}")
    y = 30
    for line in lines:
        cv2.putText(out, line, (20, y), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
        y += 30
    return out


def export_to_json(detections: list[Detection], path: Path):
    data = [{"class_id": d.class_id, "class_name": d.class_name,
             "confidence": float(d.confidence),
             "bbox": list(d.box.xyxy)} for d in detections]
    with open(path, "w") as f:
        json.dump(data, f, indent=2)


def export_to_csv(detections: list[Detection], path: Path):
    with open(path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["class_id", "class_name", "confidence", "xmin", "ymin", "xmax", "ymax"])
        for d in detections:
            writer.writerow([d.class_id, d.class_name, f"{d.confidence:.4f}",
                             d.box.xmin, d.box.ymin, d.box.xmax, d.box.ymax])


print("Helper functions defined.")

In [ ]:
# ============================================================
# Cell 4: Configuration
# ============================================================
import yaml

# --- EDIT: paths ---
DATASET_YAML = Path("/kaggle/input/aeronetra-visdrone-yolo/visdrone_yolo/dataset.yaml")
WEIGHTS_DIR = Path("/kaggle/input/aeronetra-trained-weights/best_weights")

MODELS = {
    "YOLOv8n": WEIGHTS_DIR / "yolov8n_visdrone_best.pt",
    "YOLO11n": WEIGHTS_DIR / "yolo11n_visdrone_best.pt",
    "RT-DETR-l": WEIGHTS_DIR / "rtdetr_l_visdrone_best.pt",
}

# Inference settings (from AeroNetra configs/inference/inference.yaml)
CONF_THRESH = 0.25
IOU_THRESH = 0.45
IMAGE_SIZE = 640
NUM_TEST_IMAGES = 20  # Number of images to run inference on

OUTPUT_DIR = Path("/kaggle/working/inference_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Load class names from dataset YAML
with open(DATASET_YAML) as f:
    ds_cfg = yaml.safe_load(f)
CLASS_NAMES = ds_cfg["names"]

# Get test images
test_img_dir = Path(ds_cfg["path"]) / ds_cfg.get("test", ds_cfg["val"])
test_images = sorted(test_img_dir.glob("*.jpg"))[:NUM_TEST_IMAGES]
print(f"Test images: {len(test_images)} from {test_img_dir}")
print(f"Classes: {CLASS_NAMES}")

In [ ]:
# ============================================================
# Cell 5: Run inference with all models
# ============================================================
from ultralytics import YOLO, RTDETR

all_results = {}  # {model_name: {image_name: ModelPrediction}}

for model_name, weights_path in MODELS.items():
    if not weights_path.exists():
        print(f"Skipping {model_name} — weights not found")
        continue

    print(f"\n{'='*60}")
    print(f"Running: {model_name}")
    print(f"{'='*60}")

    # Load model
    is_rtdetr = "rtdetr" in model_name.lower().replace("-", "")
    model = RTDETR(str(weights_path)) if is_rtdetr else YOLO(str(weights_path))

    predictions = {}
    total_time = 0.0

    for img_path in test_images:
        img = cv2.imread(str(img_path))
        if img is None:
            continue

        t0 = time.time()
        results = model.predict(
            source=img, conf=CONF_THRESH, iou=IOU_THRESH,
            device=DEVICE, verbose=False
        )
        inf_time = (time.time() - t0) * 1000
        total_time += inf_time

        # Parse results into our data structures
        detections = []
        result = results[0]
        if result.boxes is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            confs = result.boxes.conf.cpu().numpy()
            classes = result.boxes.cls.cpu().numpy()
            for box, conf, cls_id in zip(boxes, confs, classes):
                cid = int(cls_id)
                det = Detection(
                    box=BoundingBox(float(box[0]), float(box[1]), float(box[2]), float(box[3])),
                    class_id=cid,
                    class_name=CLASS_NAMES.get(cid, str(cid)),
                    confidence=float(conf),
                    source_model=model_name,
                    image_id=img_path.stem,
                )
                detections.append(det)

        img_h, img_w = img.shape[:2]
        pred = ModelPrediction(detections, img_w, img_h, inf_time)
        predictions[img_path.stem] = pred

    all_results[model_name] = predictions
    avg_time = total_time / max(len(predictions), 1)
    total_dets = sum(len(p.detections) for p in predictions.values())
    print(f"  Images:     {len(predictions)}")
    print(f"  Total dets: {total_dets}")
    print(f"  Avg time:   {avg_time:.1f} ms/image")

In [ ]:
# ============================================================
# Cell 6: Side-by-side visual comparison
# ============================================================
import matplotlib.pyplot as plt

# Pick first 5 images for visualization
vis_images = test_images[:5]
active_models = [m for m in MODELS if m in all_results]
n_models = len(active_models)

for img_path in vis_images:
    img = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, n_models + 1, figsize=(6 * (n_models + 1), 6))
    if n_models == 0:
        continue

    # Original image
    axes[0].imshow(img_rgb)
    axes[0].set_title(f"Original: {img_path.stem}", fontsize=10)
    axes[0].axis("off")

    # Model predictions
    colors = [(0, 255, 0), (255, 165, 0), (255, 0, 0)]
    for i, model_name in enumerate(active_models):
        pred = all_results[model_name].get(img_path.stem)
        if pred:
            annotated = draw_detections(img, pred.detections, color=colors[i % len(colors)])
            total, counts = count_vehicles(pred.detections)
            annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
            axes[i + 1].imshow(annotated_rgb)
            axes[i + 1].set_title(
                f"{model_name}: {total} detections\n{pred.inference_time_ms:.0f}ms",
                fontsize=10,
            )
        else:
            axes[i + 1].text(0.5, 0.5, "No results", ha="center", va="center")
        axes[i + 1].axis("off")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"comparison_{img_path.stem}.png", dpi=150)
    plt.show()

In [ ]:
# ============================================================
# Cell 7: Speed benchmark summary
# ============================================================
import pandas as pd

speed_data = []
for model_name, predictions in all_results.items():
    times = [p.inference_time_ms for p in predictions.values()]
    det_counts = [len(p.detections) for p in predictions.values()]
    speed_data.append({
        "Model": model_name,
        "Avg Time (ms)": f"{np.mean(times):.1f}",
        "Min Time (ms)": f"{np.min(times):.1f}",
        "Max Time (ms)": f"{np.max(times):.1f}",
        "Avg Detections": f"{np.mean(det_counts):.1f}",
        "Total Detections": sum(det_counts),
    })

if speed_data:
    df = pd.DataFrame(speed_data)
    print("Speed Benchmark")
    print("="*70)
    print(df.to_string(index=False))
    df.to_csv(OUTPUT_DIR / "speed_benchmark.csv", index=False)

In [ ]:
# ============================================================
# Cell 8: Export all detections
# ============================================================

for model_name, predictions in all_results.items():
    model_dir = OUTPUT_DIR / model_name.lower().replace("-", "_")
    model_dir.mkdir(parents=True, exist_ok=True)

    for img_name, pred in predictions.items():
        # Save annotated image
        img = cv2.imread(str(test_img_dir / f"{img_name}.jpg"))
        if img is not None:
            annotated = draw_detections(img, pred.detections)
            total, counts = count_vehicles(pred.detections)
            summary = CountSummary(img_name, total, counts, model_name)
            annotated = draw_count_summary(annotated, summary)
            cv2.imwrite(str(model_dir / f"{img_name}.jpg"), annotated)

        # Save detections as JSON
        export_to_json(pred.detections, model_dir / f"{img_name}.json")

print(f"All outputs saved to: {OUTPUT_DIR}")
print("\nTotal files:")
for model_name in all_results:
    model_dir = OUTPUT_DIR / model_name.lower().replace("-", "_")
    n_files = len(list(model_dir.glob("*")))
    print(f"  {model_name}: {n_files} files")

In [ ]:
# ============================================================
# Cell 9: Counting comparison
# ============================================================

print("Vehicle Counting Comparison")
print("="*70)

# Table: image × model → count
rows = []
for img_path in test_images:
    row = {"Image": img_path.stem}
    for model_name in all_results:
        pred = all_results[model_name].get(img_path.stem)
        row[model_name] = len(pred.detections) if pred else 0
    rows.append(row)

if rows:
    df_counts = pd.DataFrame(rows)
    print(df_counts.to_string(index=False))
    df_counts.to_csv(OUTPUT_DIR / "counting_comparison.csv", index=False)
    
    # Summary stats
    print("\nMean counts:")
    for model_name in all_results:
        print(f"  {model_name}: {df_counts[model_name].mean():.1f}")